In [1]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os
import pandas as pd

GTFS_CLEAN = "/content/drive/MyDrive/GTFS_CLEAN"   # adapte si besoin

all_agencies = []  # liste pour stocker tous les df

# Parcourir les sous-dossiers
for subfolder in sorted(os.listdir(GTFS_CLEAN)):
    folder_path = os.path.join(GTFS_CLEAN, subfolder)

    # vérifier que c'est un dossier
    if not os.path.isdir(folder_path):
        continue

    agency_path = os.path.join(folder_path, "agency.txt")

    # vérifier que le fichier existe
    if os.path.exists(agency_path):
        try:
            df = pd.read_csv(agency_path, encoding="utf-8", on_bad_lines="skip")
            df["source_folder"] = subfolder  # pour savoir d'où vient le fichier
            all_agencies.append(df)
            print(f"📥 agency.txt chargé depuis : {subfolder}")
        except Exception as e:
            print(f"⚠️ Erreur lecture dans {subfolder}: {e}")
    else:
        print(f"❌ Aucun agency.txt dans : {subfolder}")


📥 agency.txt chargé depuis : AISA (Bus Madrid-Aranda de Duero-Burgo de Osma)
📥 agency.txt chargé depuis : ALSA buses
📥 agency.txt chargé depuis : AUCORSA (Autobuses de Córdoba S.A.)
📥 agency.txt chargé depuis : AUTNA SL
📥 agency.txt chargé depuis : Alavabus
📥 agency.txt chargé depuis : Alvarez Travelers Coaches
📥 agency.txt chargé depuis : Ancebus
📥 agency.txt chargé depuis : Auif Irunbus (Lurraldebus)
📥 agency.txt chargé depuis : Autocares Baraza (Baraza Coaches)
📥 agency.txt chargé depuis : Autocares Rías Baixas (Rías Baixas Coaches)
📥 agency.txt chargé depuis : Autocorb Coaches
📥 agency.txt chargé depuis : Autoridad de Transporte Metropolitano del Area de Barcelona (ATM) Buses and trains in Catalonia (full version)
📥 agency.txt chargé depuis : Avanza Grupo (Ávila city bus)
📥 agency.txt chargé depuis : Avanza Grupo (Huesca city bus)
📥 agency.txt chargé depuis : Avanza Grupo (Mataró city bus)
📥 agency.txt chargé depuis : Avanza Grupo (Segovia city bus)
📥 agency.txt chargé depuis 

In [4]:
if all_agencies:
    agencies_df = pd.concat(all_agencies, ignore_index=True)
    print("\n📊 Total lignes agence :", len(agencies_df))
else:
    print("❌ Aucun fichier agency.txt trouvé.")



📊 Total lignes agence : 2372


In [6]:
# Afficher les 10 premières lignes
agencies_df.head(10)


,agency_id,agency_name,agency_url,agency_timezone,agency_lang,agency_phone,source_folder,agency_fare_url,agency_email,agency_branding_url
0,a-28005890,AISA,https://www.aisa-grupo.com/,Europe/Madrid,es,34918752018,AISA (Bus Madrid-Aranda de Duero-Burgo de Osma),NaN,NaN,NaN
1,300,ALSA,http://www.alsa.es,Europe/Madrid,es,902422242,ALSA buses,NaN,NaN,NaN
2,Aucorsa,Autobuses de Córdoba - S.A.,https://www.aucorsa.es,Europe/Madrid,es,957764676,AUCORSA (Autobuses de Córdoba S.A.),https://www.aucorsa.es/gestion_y_recarga_onlin...,NaN,NaN
3,autna,AUTNA,https://www.autna.com/,Europe/Madrid,es,0034 986 288 030,AUTNA SL,NaN,NaN,NaN
4,1,Álavabus,https://alavabus.eus/es,Europe/Madrid,es,945182060,Alavabus,https://alavabus.eus/es/tarifas-y-otros,alavabus@araba.eus,NaN
5,2,Transporte Comarcal,https://alavabus.eus/es,Europe/Madrid,es,945182060,Alavabus,https://alavabus.eus/es/tarifas-y-otros,alavabus@araba.eus,NaN
6,AUTOS ALVAREZ,AUTOS ALVAREZ DE VIAJEROS SL,http://vulpeti.com/moderniza/B49122856.zip,Europe/Madrid,es,980 62 05 01,Alvarez Travelers Coaches,NaN,NaN,NaN
7,ANCEBUS,"ANCEBUS, SL",http://www.ancebus.com/,Europe/Madrid,es,"923 480 575, 695 345 243",Ancebus,NaN,NaN,NaN
8,13,Auif,http://www.auif.es/,Europe/Madrid,es,943633111,Auif Irunbus (Lurraldebus),https://www.mugi.eus/index.php/es/tarjetas/tar...,NaN,NaN
9,1,Autocares Baraza,http://www.autocaresbaraza.com,Europe/Madrid,es,950390311,Autocares Baraza (Baraza Coaches),NaN,NaN,NaN


In [7]:
required_cols = ["agency_id", "agency_name", "agency_timezone"]

# Compter les valeurs null ou NaN
agencies_df[required_cols].isna().sum()


,0
agency_id,1
agency_name,0
agency_timezone,0


In [9]:
for col in required_cols:
    nan_count = agencies_df[col].isna().sum()
    empty_count = (agencies_df[col].astype(str).str.strip() == "").sum()

    print(f"🔹 {col} : NaN = {nan_count}, vides = {empty_count}, TOTAL = {nan_count + empty_count}")


🔹 agency_id : NaN = 1, vides = 0, TOTAL = 1
🔹 agency_name : NaN = 0, vides = 0, TOTAL = 0
🔹 agency_timezone : NaN = 0, vides = 0, TOTAL = 0


In [10]:
# Colonnes obligatoires
required_cols = ["agency_id", "agency_name", "agency_timezone"]

# Supprimer les lignes invalides
clean_df = agencies_df.dropna(subset=required_cols)

# Supprimer aussi les lignes vides ""
for col in required_cols:
    clean_df = clean_df[clean_df[col].astype(str).str.strip() != ""]


In [11]:
deleted_rows = len(agencies_df) - len(clean_df)
print(f"🚮 Lignes supprimées : {deleted_rows}")
print(f"📊 Lignes restantes : {len(clean_df)}")


🚮 Lignes supprimées : 1
📊 Lignes restantes : 2371


In [12]:
extra_cols = [
    "agency_url",
    "agency_lang",
    "agency_phone",
    "agency_fare_url",
    "agency_email",
    "agency_branding_url",
    "source_folder"
]

print("📊 Vérification des valeurs null / vides dans les colonnes supplémentaires\n")

for col in extra_cols:
    nan_count = agencies_df[col].isna().sum()
    empty_count = (agencies_df[col].astype(str).str.strip() == "").sum()
    total = nan_count + empty_count

    print(f"🔹 {col} : NaN = {nan_count}, vides = {empty_count}, TOTAL = {total}")


📊 Vérification des valeurs null / vides dans les colonnes supplémentaires

🔹 agency_url : NaN = 0, vides = 0, TOTAL = 0
🔹 agency_lang : NaN = 2040, vides = 0, TOTAL = 2040
🔹 agency_phone : NaN = 1780, vides = 0, TOTAL = 1780
🔹 agency_fare_url : NaN = 2302, vides = 0, TOTAL = 2302
🔹 agency_email : NaN = 2323, vides = 0, TOTAL = 2323
🔹 agency_branding_url : NaN = 2371, vides = 0, TOTAL = 2371
🔹 source_folder : NaN = 0, vides = 0, TOTAL = 0


In [13]:
# 📌 Colonnes à supprimer (inutile pour ton projet)
cols_to_drop = ["agency_branding_url", "agency_fare_url", "agency_email"]

# Supprimer les colonnes
agencies_df = agencies_df.drop(columns=cols_to_drop)

# Vérifier le résultat
print("Colonnes restantes :", agencies_df.columns.tolist())


Colonnes restantes : ['agency_id', 'agency_name', 'agency_url', 'agency_timezone', 'agency_lang', 'agency_phone', 'source_folder']


In [14]:
# 🔹 Colonnes facultatives
optional_cols = ["agency_lang", "agency_phone", "agency_url"]

# 1️⃣ Remplir les valeurs vides ou NaN
# - agency_lang : mettre 'es' par défaut si vide
# - agency_phone et agency_url : laisser NaN
import numpy as np

agencies_df["agency_lang"] = agencies_df["agency_lang"].fillna("es")
agencies_df["agency_phone"] = agencies_df["agency_phone"].replace("", np.nan)
agencies_df["agency_url"] = agencies_df["agency_url"].replace("", np.nan)

# 2️⃣ Nettoyer les numéros de téléphone
# Supprimer tous les caractères non numériques sauf '+' au début
import re

def clean_phone(phone):
    if pd.isna(phone):
        return np.nan
    phone = str(phone).strip()
    phone = re.sub(r"[^\d+]", "", phone)
    return phone

agencies_df["agency_phone"] = agencies_df["agency_phone"].apply(clean_phone)

# 3️⃣ Vérifier le résultat
print(agencies_df.head())


    agency_id                  agency_name                   agency_url  \
0  a-28005890                         AISA  https://www.aisa-grupo.com/   
1         300                         ALSA           http://www.alsa.es   
2     Aucorsa  Autobuses de Córdoba - S.A.       https://www.aucorsa.es   
3       autna                        AUTNA       https://www.autna.com/   
4           1                     Álavabus      https://alavabus.eus/es   

  agency_timezone agency_lang   agency_phone  \
0   Europe/Madrid          es    34918752018   
1   Europe/Madrid          es      902422242   
2   Europe/Madrid          es      957764676   
3   Europe/Madrid          es  0034986288030   
4   Europe/Madrid          es      945182060   

                                     source_folder  
0  AISA (Bus Madrid-Aranda de Duero-Burgo de Osma)  
1                                       ALSA buses  
2             AUCORSA (Autobuses de Córdoba S.A.)  
3                                         AUTNA 

In [16]:
# 🔹 Nettoyage avancé des colonnes obligatoires
def clean_obligatory_columns(df):
    # agency_id : minuscule, strip
    df["agency_id"] = df["agency_id"].astype(str).str.strip().str.lower()

    # agency_name : strip, optionnel title()
    df["agency_name"] = df["agency_name"].astype(str).str.strip().str.title().str.lower()

    # agency_timezone : strip, minuscule
    df["agency_timezone"] = df["agency_timezone"].astype(str).str.strip().str.lower()

    return df

agencies_df = clean_obligatory_columns(agencies_df)

# Vérifier le résultat
print(agencies_df.head())


    agency_id                  agency_name                   agency_url  \
0  a-28005890                         aisa  https://www.aisa-grupo.com/   
1         300                         alsa           http://www.alsa.es   
2     aucorsa  autobuses de córdoba - s.a.       https://www.aucorsa.es   
3       autna                        autna       https://www.autna.com/   
4           1                     álavabus      https://alavabus.eus/es   

  agency_timezone agency_lang   agency_phone  \
0   europe/madrid          es    34918752018   
1   europe/madrid          es      902422242   
2   europe/madrid          es      957764676   
3   europe/madrid          es  0034986288030   
4   europe/madrid          es      945182060   

                                     source_folder  
0  AISA (Bus Madrid-Aranda de Duero-Burgo de Osma)  
1                                       ALSA buses  
2             AUCORSA (Autobuses de Córdoba S.A.)  
3                                         AUTNA 

In [17]:
# Convertir toutes les langues en minuscules
agencies_df['agency_lang'] = agencies_df['agency_lang'].str.lower()


In [18]:
cols_order = ['agency_id', 'agency_name', 'agency_timezone',
              'agency_lang', 'agency_phone', 'agency_url', 'source_folder']
agencies_df = agencies_df[cols_order]


In [19]:
agencies_df["agency_id"] = agencies_df["agency_id"].astype(str)


In [20]:
# Garder uniquement les chiffres
agencies_df["agency_phone"] = agencies_df["agency_phone"].astype(str).str.replace(r"\D", "", regex=True)

# Garder seulement les numéros de 9 chiffres
agencies_df["agency_phone"] = agencies_df["agency_phone"].apply(
    lambda x: x if len(x) == 9 else None
)


In [21]:
import numpy as np

# 1. Convertir en string et enlever tout sauf les chiffres
agencies_df["agency_phone"] = agencies_df["agency_phone"].astype(str).str.replace(r"\D", "", regex=True)

# 2. Remplacer "nan", "" et "None" par NaN réel
agencies_df["agency_phone"].replace(["", "nan", "None"], np.nan, inplace=True)

# 3. Garder seulement les numéros de 9 chiffres, sinon NaN
agencies_df["agency_phone"] = agencies_df["agency_phone"].apply(
    lambda x: x if isinstance(x, str) and len(x) == 9 else np.nan
)


/tmp/ipython-input-1045780983.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  agencies_df["agency_phone"].replace(["", "nan", "None"], np.nan, inplace=True)


In [22]:
# Repérer les doublons exacts
doublons = agencies_df[agencies_df.duplicated(keep=False)]

# Afficher les lignes doublées
print(doublons)


Empty DataFrame
Columns: [agency_id, agency_name, agency_timezone, agency_lang, agency_phone, agency_url, source_folder]
Index: []


In [23]:
# Afficher les 10 premières lignes
agencies_df.head(10)


,agency_id,agency_name,agency_timezone,agency_lang,agency_phone,agency_url,source_folder
0,a-28005890,aisa,europe/madrid,es,NaN,https://www.aisa-grupo.com/,AISA (Bus Madrid-Aranda de Duero-Burgo de Osma)
1,300,alsa,europe/madrid,es,902422242,http://www.alsa.es,ALSA buses
2,aucorsa,autobuses de córdoba - s.a.,europe/madrid,es,957764676,https://www.aucorsa.es,AUCORSA (Autobuses de Córdoba S.A.)
3,autna,autna,europe/madrid,es,NaN,https://www.autna.com/,AUTNA SL
4,1,álavabus,europe/madrid,es,945182060,https://alavabus.eus/es,Alavabus
5,2,transporte comarcal,europe/madrid,es,945182060,https://alavabus.eus/es,Alavabus
6,autos alvarez,autos alvarez de viajeros sl,europe/madrid,es,980620501,http://vulpeti.com/moderniza/B49122856.zip,Alvarez Travelers Coaches
7,ancebus,"ancebus, sl",europe/madrid,es,NaN,http://www.ancebus.com/,Ancebus
8,13,auif,europe/madrid,es,943633111,http://www.auif.es/,Auif Irunbus (Lurraldebus)
9,1,autocares baraza,europe/madrid,es,950390311,http://www.autocaresbaraza.com,Autocares Baraza (Baraza Coaches)


In [24]:
import os

# Dossier principal contenant les sous-dossiers
main_folder = "/content/drive/MyDrive/GTFS_CLEAN"  # mettre ton chemin exact

# Parcourir toutes les valeurs uniques de source_folder
for subfolder_name in agencies_df["source_folder"].unique():
    subfolder_path = os.path.join(main_folder, subfolder_name)

    if not os.path.exists(subfolder_path):
        os.makedirs(subfolder_path)

    # Sélectionner uniquement les lignes correspondant à ce sous-dossier
    df_subset = agencies_df[agencies_df["source_folder"] == subfolder_name].copy()

    # Supprimer la colonne source_folder
    if "source_folder" in df_subset.columns:
        df_subset.drop(columns=["source_folder"], inplace=True)

    # Définir le chemin du fichier
    output_file = os.path.join(subfolder_path, "agency_clean.txt")

    # Sauvegarder le dataframe filtré
    df_subset.to_csv(output_file, index=False, sep=",", encoding="utf-8")

    print(f"📥 Fichier créé : {output_file} ({len(df_subset)} lignes)")


📥 Fichier créé : /content/drive/MyDrive/GTFS_CLEAN/AISA (Bus Madrid-Aranda de Duero-Burgo de Osma)/agency_clean.txt (1 lignes)
📥 Fichier créé : /content/drive/MyDrive/GTFS_CLEAN/ALSA buses/agency_clean.txt (1 lignes)
📥 Fichier créé : /content/drive/MyDrive/GTFS_CLEAN/AUCORSA (Autobuses de Córdoba S.A.)/agency_clean.txt (1 lignes)
📥 Fichier créé : /content/drive/MyDrive/GTFS_CLEAN/AUTNA SL/agency_clean.txt (1 lignes)
📥 Fichier créé : /content/drive/MyDrive/GTFS_CLEAN/Alavabus/agency_clean.txt (2 lignes)
📥 Fichier créé : /content/drive/MyDrive/GTFS_CLEAN/Alvarez Travelers Coaches/agency_clean.txt (1 lignes)
📥 Fichier créé : /content/drive/MyDrive/GTFS_CLEAN/Ancebus/agency_clean.txt (1 lignes)
📥 Fichier créé : /content/drive/MyDrive/GTFS_CLEAN/Auif Irunbus (Lurraldebus)/agency_clean.txt (1 lignes)
📥 Fichier créé : /content/drive/MyDrive/GTFS_CLEAN/Autocares Baraza (Baraza Coaches)/agency_clean.txt (1 lignes)
📥 Fichier créé : /content/drive/MyDrive/GTFS_CLEAN/Autocares Rías Baixas (Rías

In [25]:
import os
import pandas as pd

# Dossier principal contenant les sous-dossiers
GTFS_CLEAN = "/content/drive/MyDrive/GTFS_CLEAN"  # mettre ton chemin exact

all_stops = []  # liste pour stocker tous les df stops.txt

# Parcourir les sous-dossiers
for subfolder in sorted(os.listdir(GTFS_CLEAN)):
    folder_path = os.path.join(GTFS_CLEAN, subfolder)

    # Vérifier que c'est bien un dossier
    if not os.path.isdir(folder_path):
        continue

    stops_path = os.path.join(folder_path, "stops.txt")

    # Vérifier que le fichier stops.txt existe
    if os.path.exists(stops_path):
        try:
            df = pd.read_csv(stops_path, encoding="utf-8", on_bad_lines="skip")

            # Ajouter la colonne pour savoir d'où vient la ligne
            df["source_folder"] = subfolder

            # Ajouter le dataframe à la liste
            all_stops.append(df)

            print(f"📥 stops.txt chargé depuis : {subfolder} ({len(df)} lignes)")
        except Exception as e:
            print(f"⚠️ Erreur lecture dans {subfolder}: {e}")
    else:
        print(f"❌ Aucun stops.txt dans : {subfolder}")

# Combiner tous les dataframes en un seul si nécessaire
if all_stops:
    stops_df = pd.concat(all_stops, ignore_index=True)
    print(f"\n✅ stops.txt combinés : {len(stops_df)} lignes au total")
else:
    print("\n❌ Aucun stops.txt trouvé dans tous les sous-dossiers")


📥 stops.txt chargé depuis : AISA (Bus Madrid-Aranda de Duero-Burgo de Osma) (36 lignes)
📥 stops.txt chargé depuis : ALSA buses (11470 lignes)
📥 stops.txt chargé depuis : AUCORSA (Autobuses de Córdoba S.A.) (613 lignes)
📥 stops.txt chargé depuis : AUTNA SL (9 lignes)
📥 stops.txt chargé depuis : Alavabus (678 lignes)
📥 stops.txt chargé depuis : Alvarez Travelers Coaches (65 lignes)
📥 stops.txt chargé depuis : Ancebus (26 lignes)
📥 stops.txt chargé depuis : Auif Irunbus (Lurraldebus) (72 lignes)
📥 stops.txt chargé depuis : Autocares Baraza (Baraza Coaches) (50 lignes)
📥 stops.txt chargé depuis : Autocares Rías Baixas (Rías Baixas Coaches) (2549 lignes)
📥 stops.txt chargé depuis : Autocorb Coaches (170 lignes)
📥 stops.txt chargé depuis : Autoridad de Transporte Metropolitano del Area de Barcelona (ATM) Buses and trains in Catalonia (full version) (28410 lignes)
📥 stops.txt chargé depuis : Avanza Grupo (Ávila city bus) (194 lignes)
📥 stops.txt chargé depuis : Avanza Grupo (Huesca city b

/tmp/ipython-input-4005143002.py:38: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  stops_df = pd.concat(all_stops, ignore_index=True)


In [26]:
stops_df.sample(10)


,stop_id,stop_code,stop_name,stop_lat,stop_lon,zone_id,stop_url,location_type,parent_station,stop_timezone,source_folder,stop_desc,wheelchair_boarding,tts_stop_name,level_id,platform_code,out_of_service
51921,1147,48002030,SANTA JULIANA (1147),43.318133,-3.079989,3,NaN,0.0,NaN,Europe/Madrid,Bizkaibus,NaN,1.0,NaN,0,NaN,NaN
202832,MOBIITI:StopPlace:76305,NaN,Fenêtre,45.754772,-0.635868,NaN,NaN,1.0,NaN,NaN,Nouvelle-Aquitaine Mobilités,NaN,2.0,NaN,NaN,NaN,NaN
124851,de:08111:6091_Parent,NaN,Obertürkheim,48.761752,9.268170,NaN,NaN,1.0,NaN,NaN,NVBW - Nahverkehrsgesellschaft Baden-Württemb...,NaN,NaN,NaN,NaN,NaN,NaN
4978,0006209600000001,NaN,Isaac Peral(Esc.Ofi.Idionas)-P,42.002429,-4.522089,NaN,NaN,0.0,NaN,Europe/Madrid,ALSA buses,NaN,NaN,NaN,NaN,NaN,NaN
236597,4600866,NaN,PRADINES - Les Châtaigneraies,44.468103,1.405886,NaN,NaN,0.0,46S00866,NaN,Réseau interurbain liO Occitanie,arrêt commercial,0.0,NaN,NaN,NaN,NaN
190490,MOBIITI:StopPlace:47213,NaN,Alfred de Vigny,44.830231,-0.633958,NaN,NaN,1.0,NaN,NaN,Nouvelle-Aquitaine Mobilités,NaN,1.0,NaN,NaN,NaN,NaN
146407,de:08221:1104:0:11,NaN,"Ziegelhausen, Hirtenaue",49.426302,8.759796,NaN,NaN,NaN,NaN,NaN,NVBW - Nahverkehrsgesellschaft Baden-Württemb...,NaN,0.0,NaN,NaN,NaN,NaN
263797,SP34787,1007757,Reboreda (Boca Do Valo),42.273337,-8.594190,36045-66,NaN,NaN,NaN,NaN,Xunta de Galicia Buses,NaN,NaN,NaN,NaN,NaN,NaN
170786,de:08417:30524:0:2,NaN,Tailfingen Neuweiler Rad,48.262266,9.054272,NaN,NaN,NaN,NaN,NaN,NVBW - Nahverkehrsgesellschaft Baden-Württemb...,NaN,0.0,NaN,NaN,NaN,NaN
114687,6,NaN,"León y Castillo, 13",28.109872,-15.418223,NaN,NaN,NaN,NaN,NaN,Guaguas Municipales,NaN,NaN,NaN,NaN,NaN,NaN


In [27]:
import pandas as pd

# Colonnes obligatoires
mandatory_cols = ["stop_id", "stop_name", "stop_lat", "stop_lon"]

# Fonction pour compter les "vraies" valeurs manquantes
def count_missing(series):
    # Convertir en str pour détecter les espaces
    return series.isna().sum() + (series.astype(str).str.strip() == "").sum()

# Appliquer la fonction à chaque colonne obligatoire
missing_summary = {col: count_missing(stops_df[col]) for col in mandatory_cols}

# Afficher le résultat
print("Nombre total de valeurs manquantes, NaN ou vides par colonne obligatoire :\n")
for col, count in missing_summary.items():
    print(f"{col} : {count}")


Nombre total de valeurs manquantes, NaN ou vides par colonne obligatoire :

stop_id : 0
stop_name : 0
stop_lat : 0
stop_lon : 0


In [28]:
# Toutes les colonnes sauf les obligatoires
extra_cols = [col for col in stops_df.columns if col not in ["stop_id", "stop_name", "stop_lat", "stop_lon"]]

# Fonction pour compter les valeurs manquantes, NaN ou vides
def count_missing(series):
    return series.isna().sum() + (series.astype(str).str.strip() == "").sum()

# Nombre total de lignes dans le dataframe
total_rows = len(stops_df)

# Appliquer la fonction à toutes les colonnes supplémentaires et calculer le pourcentage
missing_summary_extra = {col: (count_missing(stops_df[col]), round(count_missing(stops_df[col])/total_rows*100, 2)) for col in extra_cols}

# Afficher le résultat de façon lisible
print("Nombre et pourcentage de valeurs manquantes, NaN ou vides par colonne supplémentaire :\n")
for col, (count, pct) in missing_summary_extra.items():
    print(f"{col} : {count} lignes manquantes ({pct}%)")


Nombre et pourcentage de valeurs manquantes, NaN ou vides par colonne supplémentaire :

stop_code : 145457 lignes manquantes (53.58%)
zone_id : 214207 lignes manquantes (78.9%)
stop_url : 230789 lignes manquantes (85.0%)
location_type : 102736 lignes manquantes (37.84%)
parent_station : 226541 lignes manquantes (83.44%)
stop_timezone : 238362 lignes manquantes (87.79%)
source_folder : 0 lignes manquantes (0.0%)
stop_desc : 209435 lignes manquantes (77.14%)
wheelchair_boarding : 72459 lignes manquantes (26.69%)
tts_stop_name : 270823 lignes manquantes (99.75%)
level_id : 267721 lignes manquantes (98.61%)
platform_code : 263779 lignes manquantes (97.16%)
out_of_service : 266686 lignes manquantes (98.23%)


In [29]:
len(stops_df)


271501

In [30]:
import pandas as pd

# Pourcentage limite pour supprimer la colonne
threshold = 0.7

# Calculer le pourcentage de valeurs manquantes (NaN ou vides) par colonne
def missing_percentage(series):
    total = len(series)
    missing = series.isna().sum() + (series.astype(str).str.strip() == "").sum()
    return missing / total

# Colonnes à supprimer si plus de 70 % manquantes, sauf stop_timezone
cols_to_drop = [col for col in stops_df.columns
                if col != "stop_timezone" and missing_percentage(stops_df[col]) > threshold]

# Supprimer les colonnes
stops_df.drop(columns=cols_to_drop, inplace=True)
print(f"Colonnes supprimées (>70% manquantes) : {cols_to_drop}")

# Remplacer les NaN ou vides dans stop_timezone par 'Europe/Madrid'
stops_df["stop_timezone"] = stops_df["stop_timezone"].replace(to_replace=[None, "", "nan", "NaN"], value="Europe/Madrid")
stops_df["stop_timezone"].fillna("Europe/Madrid", inplace=True)

print("✅ Colonnes nettoyées avec succès")


Colonnes supprimées (>70% manquantes) : ['zone_id', 'stop_url', 'parent_station', 'stop_desc', 'tts_stop_name', 'level_id', 'platform_code', 'out_of_service']
✅ Colonnes nettoyées avec succès


/tmp/ipython-input-3486416132.py:22: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  stops_df["stop_timezone"].fillna("Europe/Madrid", inplace=True)


In [31]:
stops_df.sample(10)

,stop_id,stop_code,stop_name,stop_lat,stop_lon,location_type,stop_timezone,source_folder,wheelchair_boarding
9067,0003072000000001,NaN,Ll.I. El Pi - Plz Olivera-Sols,41.976917,1.555571,0.0,Europe/Madrid,ALSA buses,NaN
187052,MOBIITI:Quay:114795,36801,Pré des Chambauds,45.722263,0.163153,0.0,Europe/Madrid,Nouvelle-Aquitaine Mobilités,2.0
216015,MOBIITI:Quay:74409,231,Mairie de Billère,43.301479,-0.397078,0.0,Europe/Madrid,Nouvelle-Aquitaine Mobilités,2.0
179687,de:09677:28592:0:1,NaN,Lengfurt Wasenberg,49.818127,9.599702,NaN,Europe/Madrid,NVBW - Nahverkehrsgesellschaft Baden-Württemb...,0.0
240163,556,613,Can Clavos (613),38.957623,1.445712,NaN,Europe/Madrid,TIB Transports of the Balearic Islands - CIE C...,NaN
267375,SP39058,5012528,CORTIÑAS,42.750241,-7.527770,NaN,Europe/Madrid,Xunta de Galicia Buses,NaN
73355,TMB_2.2323.682500,2323,Pl Miquel Casablancas,41.439710,2.186420,0.0,Europe/Madrid,Catalonia Area de Barcelona,0.0
22148,CAS_7257,3462,PL. DE LA VILA/PG. DE LES MONGES,41.574707,2.483657,0.0,Europe/Madrid,Autoridad de Transporte Metropolitano del Area...,0.0
196141,MOBIITI:Quay:77522,7919835A,PUY LARGE,46.746693,-0.588237,0.0,Europe/Madrid,Nouvelle-Aquitaine Mobilités,2.0
19644,AMB_112243,112243,Jardins de la Pau,41.321290,2.097317,0.0,Europe/Madrid,Autoridad de Transporte Metropolitano del Area...,0.0


In [33]:
stops_df["wheelchair_boarding"].value_counts(dropna=False)


,count
wheelchair_boarding,
0.0,130956
NaN,72459
2.0,48945
1.0,19141


In [35]:
stops_df["wheelchair_boarding"] = stops_df["wheelchair_boarding"].fillna(0)


In [36]:
stops_df["wheelchair_boarding"].value_counts(dropna=False)

,count
wheelchair_boarding,
0.0,203415
2.0,48945
1.0,19141


In [38]:
stops_df["wheelchair_boarding"] = stops_df["wheelchair_boarding"].astype(int)


In [39]:
# Normaliser stop_id en minuscules
stops_df['stop_id'] = stops_df['stop_id'].astype(str).str.lower()


In [40]:
# Mettre en minuscules et nettoyer les espaces multiples
stops_df["stop_name"] = stops_df["stop_name"].astype(str)  # s'assurer que c'est du texte
stops_df["stop_name"] = stops_df["stop_name"].str.lower()  # tout en minuscules
stops_df["stop_name"] = stops_df["stop_name"].str.strip()  # enlever espaces au début et à la fin
stops_df["stop_name"] = stops_df["stop_name"].str.replace(r"\s+", " ", regex=True)  # remplacer plusieurs espaces par un seul


In [41]:
stops_df["stop_id"] = stops_df["stop_id"].str.lower()

In [42]:
stops_df["stop_id"] = stops_df["stop_id"].str.strip()

In [43]:
stops_df.sample(10)

,stop_id,stop_code,stop_name,stop_lat,stop_lon,location_type,stop_timezone,source_folder,wheelchair_boarding
230305,30s14558,NaN,saint-dionisy - le soldat,43.804531,4.224458,1.0,Europe/Madrid,Réseau interurbain liO Occitanie,0
138850,de:08128:12914:0:rio,NaN,"niederstetten, ziegelmühle",49.401768,9.919790,NaN,Europe/Madrid,NVBW - Nahverkehrsgesellschaft Baden-Württemb...,0
217173,mobiiti:stopplace:77103,NaN,lavoisier,45.180931,0.743143,1.0,Europe/Madrid,Nouvelle-Aquitaine Mobilités,2
16089,als_0000897700000001,NaN,vilanova de la sal,41.878124,0.785973,0.0,Europe/Madrid,Autoridad de Transporte Metropolitano del Area...,0
229985,1006413,NaN,puimisson - pierre plantee,43.439663,3.212515,1.0,Europe/Madrid,Réseau interurbain liO Occitanie,0
239480,1574,31056,llucmajor est 1 (31056),39.488796,2.898646,NaN,Europe/Madrid,TIB Transports de les Illes Balears - CTM Tran...,0
140991,de:08136:3415:0:2,NaN,"lindach, paulushaus",48.837864,9.843766,NaN,Europe/Madrid,NVBW - Nahverkehrsgesellschaft Baden-Württemb...,0
131304,de:08118:7440:0:4,NaN,bietigheim fr.-ebert-straße,48.962737,9.141885,NaN,Europe/Madrid,NVBW - Nahverkehrsgesellschaft Baden-Württemb...,0
246316,sp12471,1002037,as sinas,42.576394,-8.817147,NaN,Europe/Madrid,Xunta de Galicia Buses,0
33881,olo_62,NaN,ceip st. roc,42.170719,2.469550,0.0,Europe/Madrid,Autoridad de Transporte Metropolitano del Area...,0


In [44]:
# Remplacer les NaN et vides par 0
stops_df['location_type'] = stops_df['location_type'].fillna(0)

# Supprimer les espaces éventuels et convertir en entier
stops_df['location_type'] = stops_df['location_type'].astype(str).str.strip()

# Remplacer les valeurs non numériques par 0
stops_df['location_type'] = stops_df['location_type'].apply(lambda x: x if x.isdigit() else '0')

# Convertir en entier
stops_df['location_type'] = stops_df['location_type'].astype(int)

# Vérifier les valeurs uniques pour s'assurer qu'elles sont correctes
print("Valeurs uniques dans location_type :", stops_df['location_type'].unique())


Valeurs uniques dans location_type : [0]


In [45]:
# --- stop_timezone en minuscules ---
stops_df["stop_timezone"] = stops_df["stop_timezone"].str.lower()

# --- wheelchair_boarding et location_type en entier ---
stops_df["wheelchair_boarding"] = stops_df["wheelchair_boarding"].fillna(0).astype(int)
stops_df["location_type"] = stops_df["location_type"].fillna(0).astype(int)

# Vérification rapide
print(stops_df[["stop_timezone", "wheelchair_boarding", "location_type"]].head())


   stop_timezone  wheelchair_boarding  location_type
0  europe/madrid                    0              0
1  europe/madrid                    0              0
2  europe/madrid                    0              0
3  europe/madrid                    0              0
4  europe/madrid                    0              0


In [46]:
stops_df.sample(10)

,stop_id,stop_code,stop_name,stop_lat,stop_lon,location_type,stop_timezone,source_folder,wheelchair_boarding
222615,mobiiti:quay:113024,11296B,peybois,44.938915,-0.633153,0,europe/madrid,Nouvelle-Aquitaine Mobilités,2
112534,pf43172000,NaN,c. major s/n (marquesina bàscula municipal),41.336808,1.176749,0,europe/madrid,Generalitat of Catalonia (Intercity bus),0
198806,mobiiti:stopplace:60806,NaN,etang de cieux,45.987339,1.049767,0,europe/madrid,Nouvelle-Aquitaine Mobilités,2
260657,sp30914,1034012,granja piroño,42.363572,-8.056043,0,europe/madrid,Xunta de Galicia Buses,0
217368,mobiiti:stopplace:77153,NaN,nouvelle du port,45.182793,0.705279,0,europe/madrid,Nouvelle-Aquitaine Mobilités,2
220910,mobiiti:stopplace:70759,NaN,benaben,45.297150,-0.927625,0,europe/madrid,Nouvelle-Aquitaine Mobilités,2
107933,pf08072090,NaN,sol ixent,41.418324,1.898071,0,europe/madrid,Generalitat of Catalonia (Intercity bus),0
91004,par_8_07716,7716,móstoles-austria,40.289005,-3.802690,0,europe/madrid,Consorcio Regional de Transportes de Madrid CR...,2
93055,par_8_18305,18305,encina-san nicasio,40.337048,-3.775604,0,europe/madrid,Consorcio Regional de Transportes de Madrid CR...,0
150297,de:08226:4252:30:west2,NaN,"wiesloch-walldorf, bf bstg west2",49.291111,8.663748,0,europe/madrid,NVBW - Nahverkehrsgesellschaft Baden-Württemb...,0


In [47]:
stops_df["stop_name"] = stops_df["stop_name"].str.replace(r"\s+", " ", regex=True).str.strip()

In [48]:
stops_df.sample(10)

,stop_id,stop_code,stop_name,stop_lat,stop_lon,location_type,stop_timezone,source_folder,wheelchair_boarding
209367,mobiiti:stopplace:59928,NaN,lr st aubin de ca-la mouline (2),44.680508,0.500080,0,europe/madrid,Nouvelle-Aquitaine Mobilités,2
110279,pf08175012,NaN,can vidal (a) mb0623,41.945616,1.880053,0,europe/madrid,Generalitat of Catalonia (Intercity bus),0
269042,sp40885,1042093,bermello,43.141673,-8.929243,0,europe/madrid,Xunta de Galicia Buses,0
112225,pf17216000,NaN,viladasens,42.095568,2.930095,0,europe/madrid,Generalitat of Catalonia (Intercity bus),0
109244,pf25217016,NaN,c/ del nord cantonada c/ enric de càrcer,41.651376,1.141219,0,europe/madrid,Generalitat of Catalonia (Intercity bus),0
78107,par_8_10652,10652,vda.yeguas-av.guijar,40.316635,-3.463995,0,europe/madrid,Consorcio Regional de Transportes de Madrid CR...,2
32303,gen_pf43166002,NaN,c. del morell,41.208813,1.205581,0,europe/madrid,Autoridad de Transporte Metropolitano del Area...,0
21517,bgs_3886,3886,institut moianès,41.806126,2.099659,0,europe/madrid,Autoridad de Transporte Metropolitano del Area...,0
200772,mobiiti:quay:83016,Tpofi2,pôle fiduciaire - centre de publipostage,44.613960,-1.120146,0,europe/madrid,Nouvelle-Aquitaine Mobilités,2
164471,de:08335:2916:0:1,NaN,stockach berlingerweg - siedlung,47.866392,9.024493,0,europe/madrid,NVBW - Nahverkehrsgesellschaft Baden-Württemb...,0


In [49]:
import os

# Dossier principal contenant les sous-dossiers
main_folder = "/content/drive/MyDrive/GTFS_CLEAN"  # mettre ton chemin exact

# Parcourir toutes les valeurs uniques de source_folder dans stops_df
for subfolder_name in stops_df["source_folder"].unique():
    subfolder_path = os.path.join(main_folder, subfolder_name)

    # Créer le dossier s'il n'existe pas
    if not os.path.exists(subfolder_path):
        os.makedirs(subfolder_path)

    # Sélectionner uniquement les lignes correspondant à ce sous-dossier
    df_subset = stops_df[stops_df["source_folder"] == subfolder_name].copy()

    # Supprimer la colonne source_folder
    if "source_folder" in df_subset.columns:
        df_subset.drop(columns=["source_folder"], inplace=True)

    # Définir le chemin du fichier
    output_file = os.path.join(subfolder_path, "stop_clean.txt")

    # Sauvegarder le dataframe filtré
    df_subset.to_csv(output_file, index=False, sep=",", encoding="utf-8")

    print(f"📥 Fichier créé : {output_file} ({len(df_subset)} lignes)")


📥 Fichier créé : /content/drive/MyDrive/GTFS_CLEAN/AISA (Bus Madrid-Aranda de Duero-Burgo de Osma)/stop_clean.txt (36 lignes)
📥 Fichier créé : /content/drive/MyDrive/GTFS_CLEAN/ALSA buses/stop_clean.txt (11470 lignes)
📥 Fichier créé : /content/drive/MyDrive/GTFS_CLEAN/AUCORSA (Autobuses de Córdoba S.A.)/stop_clean.txt (613 lignes)
📥 Fichier créé : /content/drive/MyDrive/GTFS_CLEAN/AUTNA SL/stop_clean.txt (9 lignes)
📥 Fichier créé : /content/drive/MyDrive/GTFS_CLEAN/Alavabus/stop_clean.txt (678 lignes)
📥 Fichier créé : /content/drive/MyDrive/GTFS_CLEAN/Alvarez Travelers Coaches/stop_clean.txt (65 lignes)
📥 Fichier créé : /content/drive/MyDrive/GTFS_CLEAN/Ancebus/stop_clean.txt (26 lignes)
📥 Fichier créé : /content/drive/MyDrive/GTFS_CLEAN/Auif Irunbus (Lurraldebus)/stop_clean.txt (72 lignes)
📥 Fichier créé : /content/drive/MyDrive/GTFS_CLEAN/Autocares Baraza (Baraza Coaches)/stop_clean.txt (50 lignes)
📥 Fichier créé : /content/drive/MyDrive/GTFS_CLEAN/Autocares Rías Baixas (Rías Baix

In [3]:
import os
import pandas as pd

# Dossier principal contenant les sous-dossiers
GTFS_CLEAN = "/content/drive/MyDrive/GTFS_CLEAN"  # mettre ton chemin exact

all_stop_times = []  # liste pour stocker tous les df stop_times.txt

# Parcourir les sous-dossiers
for subfolder in sorted(os.listdir(GTFS_CLEAN)):
    folder_path = os.path.join(GTFS_CLEAN, subfolder)

    # Vérifier que c'est bien un dossier
    if not os.path.isdir(folder_path):
        continue

    stop_times_path = os.path.join(folder_path, "stop_times.txt")

    # Vérifier que le fichier stop_times.txt existe
    if os.path.exists(stop_times_path):
        try:
            df = pd.read_csv(stop_times_path, encoding="utf-8", on_bad_lines="skip")

            # Ajouter la colonne pour savoir d'où vient la ligne
            df["source_folder"] = subfolder

            # Ajouter le dataframe à la liste
            all_stop_times.append(df)

            print(f"📥 stop_times.txt chargé depuis : {subfolder} ({len(df)} lignes)")
        except Exception as e:
            print(f"⚠️ Erreur lecture dans {subfolder}: {e}")
    else:
        print(f"❌ Aucun stop_times.txt dans : {subfolder}")

# Combiner tous les dataframes en un seul si nécessaire
if all_stop_times:
    stop_times_df = pd.concat(all_stop_times, ignore_index=True)
    print(f"\n✅ stop_times.txt combinés : {len(stop_times_df)} lignes au total")
else:
    print("\n❌ Aucun stop_times.txt trouvé dans tous les sous-dossiers")


📥 stop_times.txt chargé depuis : AISA (Bus Madrid-Aranda de Duero-Burgo de Osma) (340 lignes)
📥 stop_times.txt chargé depuis : ALSA buses (705664 lignes)
📥 stop_times.txt chargé depuis : AUCORSA (Autobuses de Córdoba S.A.) (128855 lignes)
📥 stop_times.txt chargé depuis : AUTNA SL (116 lignes)
📥 stop_times.txt chargé depuis : Alavabus (27904 lignes)
📥 stop_times.txt chargé depuis : Alvarez Travelers Coaches (142 lignes)
📥 stop_times.txt chargé depuis : Ancebus (52 lignes)
📥 stop_times.txt chargé depuis : Auif Irunbus (Lurraldebus) (2663 lignes)
📥 stop_times.txt chargé depuis : Autocares Baraza (Baraza Coaches) (778 lignes)
📥 stop_times.txt chargé depuis : Autocares Rías Baixas (Rías Baixas Coaches) (34312 lignes)
📥 stop_times.txt chargé depuis : Autocorb Coaches (8422 lignes)


/tmp/ipython-input-2156357455.py:22: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(stop_times_path, encoding="utf-8", on_bad_lines="skip")


📥 stop_times.txt chargé depuis : Autoridad de Transporte Metropolitano del Area de Barcelona (ATM) Buses and trains in Catalonia (full version) (4219428 lignes)
📥 stop_times.txt chargé depuis : Avanza Grupo (Ávila city bus) (22328 lignes)
📥 stop_times.txt chargé depuis : Avanza Grupo (Huesca city bus) (3916 lignes)
📥 stop_times.txt chargé depuis : Avanza Grupo (Mataró city bus) (33944 lignes)
📥 stop_times.txt chargé depuis : Avanza Grupo (Segovia city bus) (69828 lignes)
📥 stop_times.txt chargé depuis : Avanza Grupo (Soria city bus) (4387 lignes)
📥 stop_times.txt chargé depuis : Avanza Grupo (VAC-124. Huesca-Lleida with branches) (894 lignes)
📥 stop_times.txt chargé depuis : Avanza Grupo (VAC-245. Huesca-Barcelona) (342 lignes)
📥 stop_times.txt chargé depuis : Àrea Metropolitana de Barcelona (AMB) (980023 lignes)
📥 stop_times.txt chargé depuis : Bermibusa (630 lignes)
📥 stop_times.txt chargé depuis : Bizkaibus (604263 lignes)
📥 stop_times.txt chargé depuis : BlaBlaCar Bus (31610 lig

/tmp/ipython-input-2156357455.py:22: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(stop_times_path, encoding="utf-8", on_bad_lines="skip")


📥 stop_times.txt chargé depuis : Catalonia Area de Barcelona (3954736 lignes)
📥 stop_times.txt chargé depuis : City Council of Las Palmas de Gran Canaria (city bus) (155324 lignes)
📥 stop_times.txt chargé depuis : Collegamenti marittimi Grandi Navi Veloci (84 lignes)
📥 stop_times.txt chargé depuis : Collegamenti marittimi Grimaldi (60 lignes)
📥 stop_times.txt chargé depuis : Consorcio Regional de Transportes de Madrid CRTM Intercity Buses (Madrid Intercity Bus) (1255978 lignes)
📥 stop_times.txt chargé depuis : Consorcio Regional de Transportes de Madrid CRTM Light Rail Network (Red de Metro Ligero) (37336 lignes)
📥 stop_times.txt chargé depuis : Consorcio Regional de Transportes de Madrid CRTM Madrid City Bus (Autobus urbano de Madrid) (436842 lignes)
📥 stop_times.txt chargé depuis : Consorcio Regional de Transportes de Madrid Métro Ligero de Madrid (37336 lignes)
📥 stop_times.txt chargé depuis : Consorcio Regional de Transportes de Madrid Réseau de navettes CRTM (Red de Cercanías) 

...

📥 stop_times.txt chargé depuis : Nouvelle-Aquitaine Mobilités (3840484 lignes)
📥 stop_times.txt chargé depuis : Oñati urbain (Oñatiko herribusa) (922 lignes)
📥 stop_times.txt chargé depuis : Ouigo (181 lignes)
📥 stop_times.txt chargé depuis : Palma City Council (Palma de Mallorca city bus) (5431 lignes)
📥 stop_times.txt chargé depuis : Pinto City Council (895 lignes)
📥 stop_times.txt chargé depuis : Rafael Nadal Coaches (6 lignes)


/tmp/ipython-input-2156357455.py:22: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(stop_times_path, encoding="utf-8", on_bad_lines="skip")


📥 stop_times.txt chargé depuis : Réseau interurbain liO Occitanie (172366 lignes)
📥 stop_times.txt chargé depuis : Rodil (694 lignes)
📥 stop_times.txt chargé depuis : Sopela Town Hall (319 lignes)
📥 stop_times.txt chargé depuis : TIB Transports de les Illes Balears - CTM Transport Consortium of Mallorcam (Public transport on the island of Mallorca) (45551 lignes)
📥 stop_times.txt chargé depuis : TIB Transports of the Balearic Islands - CIE Consell Insular d'Eivissa (Ibiza Island Bus) (32374 lignes)
📥 stop_times.txt chargé depuis : TIB Transports of the Balearic Islands - CIME Consell Insular de Menorca (Menorca Island Bus) (9980 lignes)
📥 stop_times.txt chargé depuis : TMESA (121845 lignes)
📥 stop_times.txt chargé depuis : TRAM Alicante (57852 lignes)
📥 stop_times.txt chargé depuis : TUS (Transportes Urbanos de Santander) (118535 lignes)
📥 stop_times.txt chargé depuis : TUSSAM (Seville bus and tram) (526998 lignes)
📥 stop_times.txt chargé depuis : Tolosa City Council (Urbano de Tolosa

/tmp/ipython-input-2156357455.py:38: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  stop_times_df = pd.concat(all_stop_times, ignore_index=True)



✅ stop_times.txt combinés : 33931494 lignes au total


In [ ]:
required_cols = ["trip_id", "arrival_time", "departure_time", "stop_id", "stop_sequence"]

for col in required_cols:
    if col in stop_times_df.columns:
        null_count = stop_times_df[col].isna().sum()
        empty_count = (stop_times_df[col].astype(str).str.strip() == "").sum()
        total_missing = null_count + empty_count

        percent = (total_missing / len(stop_times_df)) * 100

        print(f"{col} : {total_missing} valeurs manquantes ({percent:.2f}%)")
    else:
        print(f"⚠️ Colonne manquante dans le DataFrame : {col}")


In [2]:
import os
import pandas as pd
from collections import defaultdict

GTFS_CLEAN = "/content/drive/MyDrive/GTFS_CLEAN"

def analyze_column(chunk, col):
    total = len(chunk)

    nan_count = chunk[col].isna().sum()
    empty_count = (chunk[col] == "").sum()
    spaces_count = chunk[col].str.isspace().sum()

    valid_series = chunk[col].dropna().astype(str)
    unique_vals = valid_series.unique()

    return nan_count, empty_count, spaces_count, total, unique_vals


print("🔍 Début de l'analyse des fichiers stop_times.txt...\n")

for subfolder in sorted(os.listdir(GTFS_CLEAN)):
    folder_path = os.path.join(GTFS_CLEAN, subfolder)
    stop_times_path = os.path.join(folder_path, "stop_times.txt")

    if not os.path.exists(stop_times_path):
        continue

    print(f"📁 Analyse du fichier : {subfolder}/stop_times.txt")

    # Dictionnaires cumulés pour les résultats
    col_stats = defaultdict(lambda: {"nan": 0, "empty": 0, "spaces": 0, "total": 0, "unique": set()})

    for chunk in pd.read_csv(stop_times_path, chunksize=500_000, dtype=str, on_bad_lines="skip"):

        for col in chunk.columns:

            nan_c, empty_c, spaces_c, total_c, unique_vals = analyze_column(chunk, col)

            col_stats[col]["nan"] += nan_c
            col_stats[col]["empty"] += empty_c
            col_stats[col]["spaces"] += spaces_c
            col_stats[col]["total"] += total_c

            # Ajouter uniques (attention : limité)
            if len(col_stats[col]["unique"]) < 50:
                col_stats[col]["unique"].update(unique_vals[:50])

    # Résultats Final
    for col, stats in col_stats.items():
        total = stats["total"]
        pct_nan = (stats["nan"] / total) * 100
        pct_empty = (stats["empty"] / total) * 100
        pct_spaces = (stats["spaces"] / total) * 100
        pct_problem = ((stats["nan"] + stats["empty"] + stats["spaces"]) / total) * 100

        print(f"\n🔎 Colonne : {col}")
        print(f"   - NaN           : {stats['nan']} ({pct_nan:.2f}%)")
        print(f"   - Vides ('')    : {stats['empty']} ({pct_empty:.2f}%)")
        print(f"   - Espaces       : {stats['spaces']} ({pct_spaces:.2f}%)")
        print(f"   - Total probs   : {pct_problem:.2f}%")
        print(f"   - Valeurs uniques (~max 50) : {len(stats['unique'])}")
        print(f"     ➤ Exemple : {list(stats['unique'])[:5]}")

    print("\n" + "-"*60 + "\n")

print("✅ Analyse terminée !")


🔍 Début de l'analyse des fichiers stop_times.txt...

📁 Analyse du fichier : AISA (Bus Madrid-Aranda de Duero-Burgo de Osma)/stop_times.txt

🔎 Colonne : trip_id
   - NaN           : 0 (0.00%)
   - Vides ('')    : 0 (0.00%)
   - Espaces       : 0 (0.00%)
   - Total probs   : 0.00%
   - Valeurs uniques (~max 50) : 20
     ➤ Exemple : ['122', '153', '151', '142', '161']

🔎 Colonne : departure_time
   - NaN           : 0 (0.00%)
   - Vides ('')    : 0 (0.00%)
   - Espaces       : 0 (0.00%)
   - Total probs   : 0.00%
   - Valeurs uniques (~max 50) : 50
     ➤ Exemple : ['13:13:00', '08:01:00', '09:02:00', '11:35:00', '07:50:00']

🔎 Colonne : arrival_time
   - NaN           : 0 (0.00%)
   - Vides ('')    : 0 (0.00%)
   - Espaces       : 0 (0.00%)
   - Total probs   : 0.00%
   - Valeurs uniques (~max 50) : 50
     ➤ Exemple : ['13:13:00', '08:01:00', '09:02:00', '11:35:00', '07:50:00']

🔎 Colonne : stop_id
   - NaN           : 0 (0.00%)
   - Vides ('')    : 0 (0.00%)
   - Espaces       : 0 (0.

/tmp/ipython-input-417830631.py:52: RuntimeWarning: invalid value encountered in scalar divide
  pct_nan = (stats["nan"] / total) * 100
/tmp/ipython-input-417830631.py:53: RuntimeWarning: invalid value encountered in scalar divide
  pct_empty = (stats["empty"] / total) * 100
/tmp/ipython-input-417830631.py:54: RuntimeWarning: invalid value encountered in scalar divide
  pct_spaces = (stats["spaces"] / total) * 100
/tmp/ipython-input-417830631.py:55: RuntimeWarning: invalid value encountered in scalar divide
  pct_problem = ((stats["nan"] + stats["empty"] + stats["spaces"]) / total) * 100



🔎 Colonne : trip_id
   - NaN           : 0 (nan%)
   - Vides ('')    : 0 (nan%)
   - Espaces       : 0 (nan%)
   - Total probs   : nan%
   - Valeurs uniques (~max 50) : 0
     ➤ Exemple : []

🔎 Colonne : arrival_time
   - NaN           : 0 (nan%)
   - Vides ('')    : 0 (nan%)
   - Espaces       : 0 (nan%)
   - Total probs   : nan%
   - Valeurs uniques (~max 50) : 0
     ➤ Exemple : []

🔎 Colonne : departure_time
   - NaN           : 0 (nan%)
   - Vides ('')    : 0 (nan%)
   - Espaces       : 0 (nan%)
   - Total probs   : nan%
   - Valeurs uniques (~max 50) : 0
     ➤ Exemple : []

🔎 Colonne : stop_id
   - NaN           : 0 (nan%)
   - Vides ('')    : 0 (nan%)
   - Espaces       : 0 (nan%)
   - Total probs   : nan%
   - Valeurs uniques (~max 50) : 0
     ➤ Exemple : []

🔎 Colonne : stop_sequence
   - NaN           : 0 (nan%)
   - Vides ('')    : 0 (nan%)
   - Espaces       : 0 (nan%)
   - Total probs   : nan%
   - Valeurs uniques (~max 50) : 0
     ➤ Exemple : []

🔎 Colonne : stop_head

/tmp/ipython-input-417830631.py:52: RuntimeWarning: invalid value encountered in scalar divide
  pct_nan = (stats["nan"] / total) * 100
/tmp/ipython-input-417830631.py:53: RuntimeWarning: invalid value encountered in scalar divide
  pct_empty = (stats["empty"] / total) * 100
/tmp/ipython-input-417830631.py:54: RuntimeWarning: invalid value encountered in scalar divide
  pct_spaces = (stats["spaces"] / total) * 100
/tmp/ipython-input-417830631.py:55: RuntimeWarning: invalid value encountered in scalar divide
  pct_problem = ((stats["nan"] + stats["empty"] + stats["spaces"]) / total) * 100



🔎 Colonne : trip_id
   - NaN           : 0 (0.00%)
   - Vides ('')    : 0 (0.00%)
   - Espaces       : 0 (0.00%)
   - Total probs   : 0.00%
   - Valeurs uniques (~max 50) : 50
     ➤ Exemple : ['5_502_15_58500', '8_802_27_72000', '4_403_25_30780', '2_204_26_37800', '3_304_25_64800']

🔎 Colonne : arrival_time
   - NaN           : 0 (0.00%)
   - Vides ('')    : 0 (0.00%)
   - Espaces       : 0 (0.00%)
   - Total probs   : 0.00%
   - Valeurs uniques (~max 50) : 50
     ➤ Exemple : ['10:30:40', '19:20:03', '19:18:08', '10:54:57', '19:30:35']

🔎 Colonne : departure_time
   - NaN           : 0 (0.00%)
   - Vides ('')    : 0 (0.00%)
   - Espaces       : 0 (0.00%)
   - Total probs   : 0.00%
   - Valeurs uniques (~max 50) : 50
     ➤ Exemple : ['10:46:45', '19:17:28', '19:20:08', '10:33:13', '19:33:08']

🔎 Colonne : stop_id
   - NaN           : 0 (0.00%)
   - Vides ('')    : 0 (0.00%)
   - Espaces       : 0 (0.00%)
   - Total probs   : 0.00%
   - Valeurs uniques (~max 50) : 50
     ➤ Exemple :